# Evidencia de ingesta de microestructura

> Cuaderno de validación reproducible del pipeline ETL de microestructura (`fact_microstructure_4h`) sobre SQL y los `parquet` raw materializados por el propio ETL.

## Contexto y motivación

La capa de microestructura es la única del proyecto que combina dos regímenes de ingesta que resultan muy distintos en su ejecución por la naturaleza de la fuente:

- Un **histórico** offline (`micro-backfill` / `micro-repair`) que reconstruye buckets 4h a partir de `aggTrades` archivados de Binance Vision y los persiste en `microstructure_data.fact_microstructure_4h`, marcando filas reales vs. imputadas con `is_imputed`.
- Un **live** continuo (`micro-ws-live`) basado en un **colector *WebSocket*** que rellena `microstructure_data.ws_agg_trade_staging` con trades en tiempo real, mantiene `microstructure_data.ws_live_state` con el estado por símbolo y, opcionalmente, **finaliza buckets 4h** y los hace UPSERT a `fact_microstructure_4h`.

A diferencia de la ingesta OHLCV (`02_evidencia_ingesta_ohlcv.ipynb`, `notebooks/01_ingesta/ohlcv/`) o de derivados (`01_evidencia_ingesta_derivados.ipynb`), las *features* de microestructura (intensidad, OFI, agresividad, etc.) son las **más sensibles al hueco**: un símbolo sin trades durante un bucket entero produce una fila imputada con valores neutros que el modelo ve como mercado plano.

Este cuaderno materializa esa evidencia en forma de:

- una **ejecución end-to-end** del validador (`scripts/validate_microstructure_pipeline.py`) con su matriz de aceptación sobre la fact `microstructure_data.fact_microstructure_4h` (no sobre staging WebSocket);
- una **batería de consultas SQL** que cuantifica cobertura, auditoría de jobs batch en `ingestion_runs`, duplicados por PK e integridad real frente a imputaciones (`is_imputed`);
- un **inventario en disco** de los `parquet` raw `{SYMBOL}_fact_microstructure_4h.parquet` materializados por el ETL histórico o `repair`;
- un **smoke test live** del colector WebSocket (`ws_agg_trade_staging`, `ws_live_state`), diseñado para no contaminar la fact agregada en la corrida por defecto;
- y la **exportación** de todo lo anterior como JSON/CSV versionados en `reports/validation/microstructure/`, para citar números trazables desde la memoria.

El flujo está pensado para ejecutarse de una vez con `Kernel > Restart Kernel and Run All`. La celda del validador hace `importlib.reload(val)` para evitar etiquetas cacheadas del kernel entre corridas; cada sección añade una pieza de evidencia y termina con interpretación o cierre.


## 1. Configuración del entorno

Se localiza la raíz del proyecto buscando hacia arriba desde el `cwd` actual hasta encontrar `scripts/validate_microstructure_pipeline.py`. Esto permite ejecutar el cuaderno tanto desde `notebooks/01_ingesta/microestructura/` como desde la raíz del repo o desde el contenedor Docker (donde la raíz es `/app`).

También se capturan los **metadatos de ejecución** (timestamp UTC, versión de Python y plataforma): sin ellos no queda registrado con claridad en qué entorno se generaron los CSV/JSON exportados al final.

In [1]:
from __future__ import annotations

import warnings

# Suprime un único aviso: Mlflow importa en cadena y dispara el UserWarning de setuptools o pkg_resources
warnings.filterwarnings(
    "ignore",
    message=".*pkg_resources is deprecated.*",
    category=UserWarning,
)

import contextlib
import importlib
import io
import json
import platform
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sqlalchemy import bindparam, inspect, text

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "validate_microstructure_pipeline.py").exists():
        ROOT = candidate
        break

assert ROOT is not None, "No se encontró la raíz del proyecto (falta scripts/validate_microstructure_pipeline.py)."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.validate_microstructure_pipeline as val
from src.config.settings import Settings
from src.data.microstructure_ws_collector import MicrostructureWsCollector
from src.utils.database import create_db_engine

SETTINGS_PATH = ROOT / "config" / "settings.yaml"
MICROSTRUCTURE_SCHEMA = "microstructure_data"
MICROSTRUCTURE_FACT_TABLE = f"{MICROSTRUCTURE_SCHEMA}.fact_microstructure_4h"
MICROSTRUCTURE_OUTPUT_DIR = ROOT / "reports" / "validation" / "microstructure"
RUN_TIMESTAMP = datetime.now(timezone.utc)
RUN_STAMP = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")
MICROSTRUCTURE_FREEZE_COVERAGE_DATE = val.MICROSTRUCTURE_FREEZE_COVERAGE_DATE

# Incorpora anulación opcional de símbolos de evidencia; None mantiene el universo del YAML
SYMBOLS_EVIDENCIA = None

settings = Settings.load_from_yaml(SETTINGS_PATH)
symbols = tuple(symbol.upper() for symbol in settings.trading_universe.symbols)
symbols_evidencia = (
    tuple(symbol.upper() for symbol in SYMBOLS_EVIDENCIA)
    if SYMBOLS_EVIDENCIA
    else symbols
)
raw_data_dir = Path(settings.microstructure_data.raw_data_dir)
raw_data_dir = raw_data_dir.resolve() if raw_data_dir.is_absolute() else (ROOT / raw_data_dir).resolve()
run_meta = {
    "fecha_hora_utc": RUN_TIMESTAMP.isoformat(),
    "raiz_proyecto": str(ROOT),
    "python": platform.python_version(),
    "plataforma": platform.platform(),
}
run_meta


{'fecha_hora_utc': '2026-06-04T00:56:44.403977+00:00',
 'raiz_proyecto': '/app',
 'python': '3.10.19',
 'plataforma': 'Linux-5.15.133.1-microsoft-standard-WSL2-x86_64-with-glibc2.36'}

## 2. Ejecución del validador end-to-end

En esta celda se invoca el validador `scripts/validate_microstructure_pipeline.py` reutilizando exactamente la misma lógica que la *pipeline* de CI, pero adaptada al contexto interactivo del cuaderno:

- Se redirige `stdout`/`stderr` a un buffer en la propia celda para no saturar la salida del cuaderno con el logging técnico del validador.
- Si la conexión a SQL falla, las comprobaciones aguas abajo se marcan como `SKIP` en lugar de lanzar una excepción, dejando un informe parcial pero coherente.
- Si las tablas `microstructure_data.*` están listas, se ejecutan en cadena: suite `integration` -> suite `db_integration` -> auditoría (`ingestion_runs`/`ingestion_events`) -> calidad mínima (`DATA_QUALITY_MIN`) -> cobertura freeze (`FREEZE_COVERAGE`) contra el bucket **2026-02-28 20:00 UTC** (`MICROSTRUCTURE_FREEZE_COVERAGE_DATE`).
- Finalmente se construye la **matriz de aceptación común** (`build_common_acceptance_matrix`) que define qué comprobaciones son obligatorias para considerar el ETL aceptado.

El resultado es la tabla `summary_df`, donde cada fila es una comprobación con su estado (`PASS`/`FAIL`/`SKIP`), si es requerida y un detalle legible. Esta tabla es el semáforo de alto nivel del ETL.


In [2]:
importlib.reload(val)

# Captura el logging técnico estructurado y mantiene la salida del cuaderno limpia
technical_logs_buffer = io.StringIO()
with contextlib.redirect_stdout(technical_logs_buffer), contextlib.redirect_stderr(technical_logs_buffer):
    val.results.clear()
    db_ready = val.check_db_connection()
    val.check_micro_tables()
    val.run_integration_suite()

    if db_ready and val.results.get("TABLES_READY") == val.PASS:
        val.run_db_integration_suite()
        val.check_audit_evidence(db_integration_passed=val.results.get("DB_INTEGRATION") == val.PASS)
        val.check_data_quality_min(symbols=symbols)
        val.check_freeze_coverage(MICROSTRUCTURE_FREEZE_COVERAGE_DATE)
    else:
        val.results["DB_INTEGRATION"] = val.SKIP
        val.set_status(val.results, check_id="AUDIT_RUNS", status=val.SKIP)
        val.set_status(val.results, check_id="AUDIT_EVENTS", status=val.SKIP, aliases=("AUDIT_EVIDENCE",))
        val.set_status(val.results, check_id="DATA_QUALITY_MIN", status=val.SKIP, aliases=("DATA_QUALITY",))
        val.results["FREEZE_COVERAGE"] = val.SKIP

    acceptance_matrix = val.build_common_acceptance_matrix(
        target_date=MICROSTRUCTURE_FREEZE_COVERAGE_DATE,
        db_ready=db_ready,
        tables_ready=val.results.get("TABLES_READY") == val.PASS,
        data_quality_required=True,
    )
    accepted, actionable_failures = val._print_summary(acceptance_matrix)

_ = technical_logs_buffer.getvalue()
print("Ejecución completada. Se ocultó el log técnico detallado.")

matrix_required = {str(row["check_id"]): bool(row["required"]) for row in acceptance_matrix}


def detalle_check(check_id: str, status: str) -> str:
    if status == val.FAIL:
        for failure in actionable_failures:
            if check_id in failure:
                return failure
        return "Fallo detectado. Revisar logs de validación para diagnóstico."

    if status == val.SKIP:
        return "No ejecutado por precondiciones no cumplidas."

    detalles_ok = {
        "INTEGRATION": "Suite integration ejecutada correctamente.",
        "DB_INTEGRATION": "Suite db_integration ejecutada correctamente.",
        "AUDIT_RUNS": "Evidencia de ejecuciones comprobada en la tabla de auditoría.",
        "AUDIT_EVENTS": "Evidencia de eventos comprobada y asociada a ejecuciones.",
        "DATA_QUALITY_MIN": "Sin duplicados críticos PK; cobertura temporal al freeze: FREEZE_COVERAGE.",
        "FREEZE_COVERAGE": (
            f"Cobertura en fact hasta el bucket freeze 20:00 UTC del "
            f"{MICROSTRUCTURE_FREEZE_COVERAGE_DATE} "
            f"(FREEZE_COVERAGE / MICROSTRUCTURE_FREEZE_COVERAGE_DATE)."
        ),
    }
    return detalles_ok.get(check_id, "Comprobación validada correctamente.")


summary_rows = []
for check_id, label in val._CHECK_LABELS.items():
    status = val.results.get(check_id, val.SKIP)
    summary_rows.append(
        {
            "id_check": check_id,
            "etiqueta": label,
            "estado": status,
            "requerido": "Sí" if matrix_required.get(check_id, False) else "No",
            "detalle": detalle_check(check_id, status),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df


Ejecución completada. Se ocultó el log técnico detallado.


,id_check,etiqueta,estado,requerido,detalle
0,DB_CONNECTION,Conexión a TimescaleDB,PASS,Sí,Comprobación validada correctamente.
1,TABLES_READY,Tablas microstructure_data disponibles,PASS,Sí,Comprobación validada correctamente.
2,INTEGRATION,Suite de integración de microestructura,PASS,Sí,Suite integration ejecutada correctamente.
3,DB_INTEGRATION,Suite de integración con BD de microestructura,PASS,Sí,Suite db_integration ejecutada correctamente.
4,AUDIT_RUNS,Evidencia en microstructure_data.ingestion_runs,PASS,Sí,Evidencia de ejecuciones comprobada en la tabl...
5,AUDIT_EVENTS,Evidencia en microstructure_data.ingestion_events,PASS,Sí,Evidencia de eventos comprobada y asociada a e...
6,DATA_QUALITY_MIN,Deduplicación PK,PASS,Sí,Sin duplicados críticos PK; cobertura temporal...
7,FREEZE_COVERAGE,Cobertura micro por símbolo hasta target bucke...,PASS,Sí,Cobertura en fact hasta el bucket freeze 20:00...


### 2.1. Lectura de la matriz de aceptación

Las comprobaciones devueltas cubren las cuatro capas críticas del ETL de microestructura:

| Capa | Comprobaciones asociadas | Por qué importa |
|---|---|---|
| **Infraestructura** | `DB_CONNECTION`, `TABLES_READY` | Sin BD viva ni la fact 4h creada, no hay nada que validar |
| **Pipeline** | `INTEGRATION`, `DB_INTEGRATION` | Las suites `integration` y `db_integration` corren de extremo a extremo: agregación de `aggTrades` + persistencia con esquema y casing correctos |
| **Calidad de datos** | `DATA_QUALITY_MIN`, `FREEZE_COVERAGE` | Sin duplicados ni violaciones de dominio/rango; cobertura por símbolo **hasta el bucket freeze 4h** de **`MICROSTRUCTURE_FREEZE_COVERAGE_DATE`** (2026-02-28 20:00 UTC) |
| **Auditoría** | `AUDIT_RUNS`, `AUDIT_EVENTS` | Cada ejecución queda trazada en `ingestion_runs`/`ingestion_events` para reconstruir lo que se ingestó y cuándo |

**En esta ejecución** (`stamp` `20260603T233430Z`, `fecha_hora_utc` **2026-06-03 23:34 UTC**), las ocho comprobaciones del validador figuran en `PASS` y el export cierra con resultado global `PASS`, lo que valida la **base mínima** sobre la que pueden empezar a construirse *features* y modelos. Si alguna comprobación aparece en `FAIL`, la columna `detalle` apunta a la pista accionable (tabla vacía, símbolo huérfano, hueco de cobertura, etc.).


## 3. Evidencia SQL: cobertura, auditoría e integridad

Más allá del semáforo del validador, esta sección aterriza la evidencia en **números concretos** sobre `microstructure_data.fact_microstructure_4h` (tabla origen del histórico; no se valida `ws_agg_trade_staging`). Se lanzan **cuatro** consultas SQL contra `microstructure_data.*`:

1. **Cobertura por símbolo** sobre la fact: `MIN(bucket_4h)`, `MAX(bucket_4h)` y conteo de filas, cruzado contra el universo esperado para detectar símbolos huérfanos.
2. **Ejecuciones batch en `ingestion_runs`:** agregación global de backfill/repair. **No** cubre el colector WebSocket: el live operativo se evidencia en la **sección 5** (`ws_agg_trade_staging`, `ws_live_state`). El detalle por `pipeline_type` está en el CSV de auditoría exportado en la sección 6.
3. **Duplicados por PK** `(symbol, bucket_4h)`: debe ser estrictamente cero por símbolo.
4. **Integridad real vs. imputaciones** mediante `is_imputed` (buckets sin trades reales rellenados con valores neutros).

Adicionalmente se construye un **inventario en disco** de los `parquet` raw `{SYMBOL}_fact_microstructure_4h.parquet` bajo `microstructure_data.raw_data_dir`. Sirve como respaldo crudo independiente de la BD: si hay que reconstruir la fact sin reprocesar `aggTrades` desde Binance Vision, el Parquet es la fuente de verdad. Este cuaderno **solo lee** ese directorio; un `tamaño_mb` vacío indica archivo ausente (reejecutar `micro-backfill` o `micro-repair`).

> **Nota técnica:** `max_timestamp` en cobertura es el máximo real en BD por símbolo (el live puede desalinear series). El cumplimiento del freeze (`FREEZE_COVERAGE`) va en el validador. Sobre `total_fallos_eventos` en auditoría: suma `failed_count` por run; un valor puntual > 0 no invalida la fact si cobertura, duplicados e integridad siguen en `PASS`.


In [3]:
cobertura_objetivo_df = pd.DataFrame()
resumen_modos_df = pd.DataFrame()
evidencia_auditoria_df = pd.DataFrame()
duplicados_pk_df = pd.DataFrame()
integridad_df = pd.DataFrame()
vista_cobertura_filas_df = pd.DataFrame(
    columns=["símbolo", "tabla", "filas", "min_timestamp", "max_timestamp"]
)

# Cohorte batch en ingestion_runs (misma que el validador de auditoría)
MICRO_BATCH_PT_SQL = (
    "'etl_microstructure', 'etl_microstructure_backfill', 'etl_microstructure_repair', "
    "'etl_microstructure_live', 'etl_microstructure_ws_live'"
)
MICRO_RESUMEN_MODOS_QUERY = text(
    f"""
    SELECT
        CASE
            WHEN LOWER(CAST(r.pipeline_type AS TEXT)) LIKE '%live%' THEN 'live'
            ELSE 'histórico'
        END AS modo,
        COUNT(*)::bigint AS ejecuciones,
        MAX(r.started_at) AS ultimo_inicio,
        COALESCE(SUM(r.total_rows), 0)::bigint AS total_filas,
        COALESCE(SUM(r.failed_count), 0)::bigint AS total_fallos_eventos,
        SUM(CASE WHEN COALESCE(r.failed_count, 0) > 0 THEN 1 ELSE 0 END)::bigint AS ejecuciones_con_fallo
    FROM {MICROSTRUCTURE_SCHEMA}.ingestion_runs r
    WHERE r.pipeline_type IN ({MICRO_BATCH_PT_SQL})
    GROUP BY 1
    ORDER BY modo;
    """
)

engine = None
if db_ready:
    db_engine_log = io.StringIO()
    with contextlib.redirect_stdout(db_engine_log), contextlib.redirect_stderr(db_engine_log):
        engine = create_db_engine()
    _ = db_engine_log.getvalue()

if db_ready and engine:
    cobertura_query = text(
        f"""
        SELECT
            symbol,
            COUNT(*) AS filas,
            MIN(bucket_4h) AS min_bucket_4h,
            MAX(bucket_4h) AS max_bucket_4h
        FROM {MICROSTRUCTURE_FACT_TABLE}
        GROUP BY symbol
        ORDER BY symbol;
        """
    )
    cobertura_objetivo_df = pd.DataFrame({"symbol": list(symbols_evidencia)}).merge(
        pd.read_sql(cobertura_query, engine),
        on="symbol",
        how="left",
    )
    cobertura_objetivo_df["filas"] = cobertura_objetivo_df["filas"].fillna(0).astype(int)

    if not cobertura_objetivo_df.empty:
        cobertura_objetivo_df["min_bucket_4h"] = pd.to_datetime(
            cobertura_objetivo_df["min_bucket_4h"], utc=True, errors="coerce"
        )
        cobertura_objetivo_df["max_bucket_4h"] = pd.to_datetime(
            cobertura_objetivo_df["max_bucket_4h"], utc=True, errors="coerce"
        )

    vista_cobertura_filas_df = cobertura_objetivo_df.assign(tabla=MICROSTRUCTURE_FACT_TABLE).rename(
        columns={"symbol": "símbolo", "min_bucket_4h": "min_timestamp", "max_bucket_4h": "max_timestamp"}
    )[["símbolo", "tabla", "filas", "min_timestamp", "max_timestamp"]]

    resumen_modos_df = pd.read_sql(MICRO_RESUMEN_MODOS_QUERY, engine)
    if not resumen_modos_df.empty:
        resumen_modos_df["pct_ejecuciones_con_fallo"] = (
            (resumen_modos_df["ejecuciones_con_fallo"] / resumen_modos_df["ejecuciones"]) * 100.0
        ).round(2)

    duplicados_query = text(
        f"""
        SELECT symbol,
               '4h'::text AS timeframe,
               COUNT(*) AS total_filas,
               COUNT(DISTINCT bucket_4h) AS timestamps_unicos,
               COUNT(*) - COUNT(DISTINCT bucket_4h) AS duplicados_pk
        FROM {MICROSTRUCTURE_FACT_TABLE}
        GROUP BY symbol
        ORDER BY symbol;
        """
    )
    duplicados_pk_df = pd.read_sql(duplicados_query, engine).rename(columns={"symbol": "símbolo"})

    integridad_query = text(
        f"""
        SELECT symbol,
               COUNT(*) AS total_filas,
               SUM(CASE WHEN is_imputed THEN 1 ELSE 0 END) AS filas_imputadas,
               SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END) AS filas_reales,
               ROUND(
                   100.0 * SUM(CASE WHEN NOT is_imputed THEN 1 ELSE 0 END)
                   / NULLIF(COUNT(*), 0), 4
               ) AS pct_integridad_real
        FROM {MICROSTRUCTURE_FACT_TABLE}
        GROUP BY symbol
        ORDER BY symbol;
        """
    )
    integridad_df = pd.read_sql(integridad_query, engine).rename(columns={"symbol": "símbolo"})

    auditoria_query = text(
        f"""
        SELECT
            r.run_id,
            r.pipeline_type,
            r.started_at,
            r.finished_at,
            r.success_count,
            r.failed_count,
            r.total_rows
        FROM {MICROSTRUCTURE_SCHEMA}.ingestion_runs r
        WHERE r.pipeline_type IN ({MICRO_BATCH_PT_SQL})
        ORDER BY r.started_at DESC
        LIMIT 50;
        """
    )
    evidencia_auditoria_df = pd.read_sql(auditoria_query, engine)

# Construye el inventario de artefactos Parquet en disco
parquet_rows: list[dict] = []
for symbol in symbols_evidencia:
    parquet_path = raw_data_dir / f"{symbol}_fact_microstructure_4h.parquet"
    parquet_record = {
        "tipo": "raw_microstructure_fact",
        "símbolo": symbol,
        "archivo": parquet_path.name,
        "tamaño_mb": None,
        "filas": None,
        "columnas": None,
        "min_timestamp": None,
        "max_timestamp": None,
    }

    if parquet_path.exists():
        parquet_record["tamaño_mb"] = round(parquet_path.stat().st_size / (1024 * 1024), 2)
        try:
            microstructure_fact_df = pd.read_parquet(parquet_path)
            timestamp_column = (
                "bucket_4h"
                if "bucket_4h" in microstructure_fact_df.columns
                else "timestamp"
                if "timestamp" in microstructure_fact_df.columns
                else None
            )
            timestamps = (
                pd.to_datetime(microstructure_fact_df[timestamp_column], utc=True, errors="coerce").dropna()
                if timestamp_column
                else pd.Series(dtype="datetime64[ns, UTC]")
            )
            parquet_record.update(
                {
                    "filas": len(microstructure_fact_df),
                    "columnas": len(microstructure_fact_df.columns),
                    "min_timestamp": timestamps.min() if not timestamps.empty else None,
                    "max_timestamp": timestamps.max() if not timestamps.empty else None,
                }
            )
        except Exception:
            pass

    parquet_rows.append(parquet_record)

parquet_inventory_df = pd.DataFrame(parquet_rows)



## 4. Resultados y análisis

A continuación se muestran en orden: cobertura sobre la fact, **resumen de jobs batch** en `ingestion_runs`, duplicados por PK, integridad/imputaciones e inventario Parquet en disco.


In [4]:
print("Resumen de jobs batch en ingestion_runs")
display(resumen_modos_df)

print("\nCobertura por símbolo (tabla origen)")
display(vista_cobertura_filas_df)

print("\nDuplicados por PK (símbolo, bucket_4h)")
display(duplicados_pk_df)
if not duplicados_pk_df.empty and (duplicados_pk_df["duplicados_pk"] == 0).all():
    print("Cero duplicados confirmados para todos los símbolos.")
else:
    print("Se detectaron duplicados o la consulta no devolvió resultados.")

print("\nIntegridad e imputaciones por símbolo")
display(integridad_df)

print("\nInventario de artefactos Parquet en disco")
display(parquet_inventory_df)



Resumen de jobs batch en ingestion_runs


,modo,ejecuciones,ultimo_inicio,total_filas,total_fallos_eventos,ejecuciones_con_fallo,pct_ejecuciones_con_fallo
0,histórico,2,2026-05-20 19:33:12.210586+00:00,87320,0,0,0.0



Cobertura por símbolo (tabla origen)


,símbolo,tabla,filas,min_timestamp,max_timestamp
0,BTCUSDT,microstructure_data.fact_microstructure_4h,19193,2017-08-17 04:00:00+00:00,2026-05-20 20:00:00+00:00
1,ETHUSDT,microstructure_data.fact_microstructure_4h,19193,2017-08-17 04:00:00+00:00,2026-05-20 20:00:00+00:00
2,BNBUSDT,microstructure_data.fact_microstructure_4h,18708,2017-11-06 00:00:00+00:00,2026-05-20 20:00:00+00:00
3,XRPUSDT,microstructure_data.fact_microstructure_4h,17632,2018-05-04 08:00:00+00:00,2026-05-20 20:00:00+00:00
4,SOLUSDT,microstructure_data.fact_microstructure_4h,12653,2020-08-11 04:00:00+00:00,2026-05-20 20:00:00+00:00



Duplicados por PK (símbolo, bucket_4h)


,símbolo,timeframe,total_filas,timestamps_unicos,duplicados_pk
0,BNBUSDT,4h,18708,18708,0
1,BTCUSDT,4h,19193,19193,0
2,ETHUSDT,4h,19193,19193,0
3,SOLUSDT,4h,12653,12653,0
4,XRPUSDT,4h,17632,17632,0


Cero duplicados confirmados para todos los símbolos.

Integridad e imputaciones por símbolo


,símbolo,total_filas,filas_imputadas,filas_reales,pct_integridad_real
0,BNBUSDT,18708,1,18707,99.9947
1,BTCUSDT,19193,1,19192,99.9948
2,ETHUSDT,19193,1,19192,99.9948
3,SOLUSDT,12653,0,12653,100.0000
4,XRPUSDT,17632,1,17631,99.9943



Inventario de artefactos Parquet en disco


,tipo,símbolo,archivo,tamaño_mb,filas,columnas,min_timestamp,max_timestamp
0,raw_microstructure_fact,BTCUSDT,BTCUSDT_fact_microstructure_4h.parquet,2.62,19193,20,2017-08-17 04:00:00+00:00,2026-05-20 20:00:00+00:00
1,raw_microstructure_fact,ETHUSDT,ETHUSDT_fact_microstructure_4h.parquet,2.62,19193,20,2017-08-17 04:00:00+00:00,2026-05-20 20:00:00+00:00
2,raw_microstructure_fact,BNBUSDT,BNBUSDT_fact_microstructure_4h.parquet,2.56,18708,20,2017-11-06 00:00:00+00:00,2026-05-20 20:00:00+00:00
3,raw_microstructure_fact,XRPUSDT,XRPUSDT_fact_microstructure_4h.parquet,2.40,17632,20,2018-05-04 08:00:00+00:00,2026-05-20 20:00:00+00:00
4,raw_microstructure_fact,SOLUSDT,SOLUSDT_fact_microstructure_4h.parquet,1.73,12653,20,2020-08-11 04:00:00+00:00,2026-05-20 20:00:00+00:00


### 4.1. Interpretación de los resultados

- **Resumen batch (`ingestion_runs`)**: en esta corrida hay **2** ejecuciones en modo `histórico` (último inicio **2026-05-20 19:33 UTC**, **87,320** filas acumuladas reportadas, **0** fallos de evento). El desglose por `pipeline_type` está en el CSV de auditoría exportado (sección 6).
- **Cobertura por símbolo**: **87,379** filas 4h agregadas en las cinco facts; `max_timestamp` común **2026-05-20 20:00 UTC** en esta corrida. El cumplimiento formal del freeze (`FREEZE_COVERAGE`) es independiente de esta tabla. Un `filas = 0` es un símbolo huérfano y exige reejecutar la ingesta.
- **Duplicados por PK**: contrato **cero estricto** sobre `(symbol, bucket_4h)`. Cualquier valor positivo invalida la PK natural.
- **Integridad e imputaciones**: `pct_integridad_real` ~**99.99 %** en BTC/ETH/BNB/XRP (**1** fila imputada por símbolo); **SOLUSDT** al **100 %** real.
- **Smoke *WebSocket* (sección 5)**: con `LIVE_WS_EMBEDDED_RUN_SEC = 1800`, esta ejecución arrancó el colector embebido (**5** símbolos, **93,740** trades en staging, limpieza posterior) sin escribir en la fact; evidencia operativa complementaria al batch histórico.
- **Inventario Parquet**: **5/5** ficheros `{SYMBOL}_fact_microstructure_4h.parquet` coherentes con cobertura SQL; respaldo para reconstruir la fact sin reprocesar *aggTrades* desde Binance Vision.


## 5. Snapshots live (smoke test del colector WebSocket)

En esta sección el objetivo es comprobar, sin contaminar el histórico, que el colector live (`MicrostructureWsCollector`) **recibe y persiste** trades en BD: `microstructure_data.ws_agg_trade_staging` (trades crudos por símbolo) y `microstructure_data.ws_live_state` (estado por símbolo: último `agg_trade_id`, último `trade_ts`, bucket activo, etc.).

Hay dos formas de validar el live:

- **Externo** — recomendado para evidencia real en producción: arrancar `python -m src.main --mode micro-ws-live` en otra terminal y, **en paralelo**, ejecutar esta celda con `RUN_LIVE_SNAPSHOT = True` y `LIVE_WS_EMBEDDED_RUN_SEC = 0`. La celda actúa como **observador** y consulta `ws_live_state` / `ws_agg_trade_staging` para todos los símbolos. En este modo el colector residente honra `enable_live_finalization: true` del YAML, así que cierra buckets 4h y aplica UPSERT hacia `fact_microstructure_4h`.
- **Embebido** — diseñado para el TFG y para CI: con `LIVE_WS_EMBEDDED_RUN_SEC > 0` (típicamente 1800 s = 30 min, tope duro 7200 s) la celda arranca el colector **en un hilo** durante esa ventana sobre un subconjunto del universo (`LIVE_WS_EMBEDDED_DEFAULT_N_SYMBOLS`, típicamente 5). Por defecto, este modo:
  - Desactiva el finalizador 4h (`LIVE_WS_EMBEDDED_DISABLE_FINALIZATION = True`) -> **no escribe en `fact_microstructure_4h`**.
  - Limpia `ws_agg_trade_staging` y `ws_live_state` solo para esos símbolos al terminar (`LIVE_WS_EMBEDDED_CLEANUP_AFTER = True`).

Esto garantiza que ejecutar el cuaderno entero con *Run All* no contamina la fact agregada ni deja residuos de smoke tests en staging.

**Criterios de éxito en modo embebido**: cada símbolo de prueba debe aparecer en `ws_agg_trade_staging` con al menos una fila tras la ventana, y el agregado debe traer `max_timestamp` no nulo. En modo externo basta con que `ws_live_state` tenga filas con `ultimo_trade_ts` no nulo para confirmar que el colector está viendo trades en esta misma BD.

En modo embebido, **mientras el hilo del colector está activo**, la celda redirige **stdout/stderr** a un buffer interno (structlog y mensajes de la pila WebSocket van ahí). Al terminar la ventana solo se muestran los resúmenes pensados para la evidencia y las tablas del snapshot.

> Si las tablas WebSocket no existen (`ws_agg_trade_staging`, `ws_live_state`), la celda lanza `RuntimeError`: hay que aplicar el DDL de live (volumen Postgres actualizado o reinicio con migraciones). Con `RUN_LIVE_SNAPSHOT = False` y `LIVE_WS_EMBEDDED_RUN_SEC = 0` la celda no abre WebSocket; con `LIVE_WS_EMBEDDED_RUN_SEC > 0` (p. ej. **1800** s en este cuaderno) sí ejecuta el smoke embebido sin escribir en `fact_microstructure_4h` y limpia staging al terminar.


In [5]:
# Fija el modo snapshot (observador) cuando un colector ya corre fuera de esta celda
RUN_LIVE_SNAPSHOT = False
LIVE_SNAPSHOT_LABEL = "antes_o_despues_prueba_8h"

# Arranca en hilo el colector mientras LIVE_WS_EMBEDDED_RUN_SEC sea mayor que 0
# Mapea 1800s a 30 min y 3600s a 1 h; aplica tope duro con LIVE_WS_MAX_EMBEDDED_SEC
LIVE_WS_EMBEDDED_RUN_SEC = 1800
LIVE_WS_MAX_EMBEDDED_SEC = 7200

# Con None, toma los N primeros símbolos del YAML solo en modo embebido
# Con lista explícita, exige secuencia no vacía
# En snapshot sin embebido, ignora el subconjunto y lee el universo completo
LIVE_WS_SYMBOLS = None
LIVE_WS_EMBEDDED_DEFAULT_N_SYMBOLS = 5

# Con embebido, desactiva el UPSERT hacia fact_microstructure_4h
LIVE_WS_EMBEDDED_DISABLE_FINALIZATION = True

# Con embebido, borra staging y ws_live_state de live_symbols al cierre
LIVE_WS_EMBEDDED_CLEANUP_AFTER = True

WS_TABLES = ("ws_live_state", "ws_agg_trade_staging")

LIVE_SNAPSHOT_COL_STAGING_ES = {
    "symbol": "símbolo",
    "rows_staging": "filas",
    "min_transact_time": "min_timestamp",
    "max_transact_time": "max_timestamp",
}
LIVE_SNAPSHOT_COL_STATE_ES = {
    "symbol": "símbolo",
    "last_agg_trade_id": "ultimo_id_agg_trade",
    "last_trade_ts": "ultimo_trade_ts",
    "active_bucket_start": "inicio_bucket_activo",
    "active_bucket_trade_count": "trades_bucket_activo",
    "last_finalized_bucket": "ultimo_bucket_finalizado",
    "last_rest_backfill_ts": "ultimo_backfill_rest",
    "updated_at": "actualizado_en",
}

micro_live_snapshot_state = pd.DataFrame()
micro_live_snapshot_staging = pd.DataFrame()

live_checks_enabled = RUN_LIVE_SNAPSHOT or LIVE_WS_EMBEDDED_RUN_SEC > 0

if not live_checks_enabled:
    print(
        "Live WS desactivado: RUN_LIVE_SNAPSHOT=False y LIVE_WS_EMBEDDED_RUN_SEC=0. "
        "Usa LIVE_WS_EMBEDDED_RUN_SEC>0 para arrancar el colector en esta celda, "
        "o RUN_LIVE_SNAPSHOT=True si micro-ws-live ya corre fuera."
    )
else:
    db_connection_log = io.StringIO()
    with contextlib.redirect_stdout(db_connection_log), contextlib.redirect_stderr(db_connection_log):
        live_engine = create_db_engine()
    if live_engine is None:
        raise RuntimeError(
            "Prueba live no superada: sin motor de BD (revisa POSTGRES_*)."
        )
    try:
        inspector = inspect(live_engine)
        missing_tables = [table for table in WS_TABLES if not inspector.has_table(table, schema=MICROSTRUCTURE_SCHEMA)]
        if missing_tables:
            raise RuntimeError(
                "Prueba live no superada: faltan tablas WebSocket en la BD: "
                + ", ".join(f"{MICROSTRUCTURE_SCHEMA}.{table}" for table in missing_tables)
                + ". Aplica el DDL de live WS (p. ej. volumen Postgres actualizado)."
            )

        if LIVE_WS_SYMBOLS is not None:
            live_symbols = [str(symbol).upper() for symbol in LIVE_WS_SYMBOLS]
            if not live_symbols:
                raise ValueError("LIVE_WS_SYMBOLS no puede ser una secuencia vacía.")
        elif LIVE_WS_EMBEDDED_RUN_SEC > 0:
            embedded_symbol_count = max(1, int(LIVE_WS_EMBEDDED_DEFAULT_N_SYMBOLS))
            live_symbols = list(symbols[: min(embedded_symbol_count, len(symbols))])
        else:
            live_symbols = list(symbols)

        if LIVE_WS_EMBEDDED_RUN_SEC > 0:
            embedded_duration_sec = min(int(LIVE_WS_EMBEDDED_RUN_SEC), LIVE_WS_MAX_EMBEDDED_SEC)
            if embedded_duration_sec != int(LIVE_WS_EMBEDDED_RUN_SEC):
                print(
                    f"Aviso: LIVE_WS_EMBEDDED_RUN_SEC recortado a {embedded_duration_sec}s (máx {LIVE_WS_MAX_EMBEDDED_SEC}s)."
                )
            live_config = settings.model_dump(mode="python")
            if LIVE_WS_EMBEDDED_DISABLE_FINALIZATION:
                live_config["microstructure_data"]["ws_live"]["enable_live_finalization"] = False
            collector = MicrostructureWsCollector(live_config)
            collector_errors: list[BaseException] = []

            def collector_thread_main() -> None:
                try:
                    collector.run(live_engine, live_symbols)
                except BaseException as exc:
                    collector_errors.append(exc)

            live_flags = []
            if LIVE_WS_EMBEDDED_DISABLE_FINALIZATION:
                live_flags.append("sin finalización 4h -> fact")
            if LIVE_WS_EMBEDDED_CLEANUP_AFTER:
                live_flags.append("limpieza staging/estado al terminar")
            live_flags_text = f" | ({'; '.join(live_flags)})" if live_flags else ""

            print(
                f"Colector WS embebido: {embedded_duration_sec}s | inicio UTC "
                f"{datetime.now(timezone.utc).isoformat()} | símbolos={live_symbols}{live_flags_text}"
            )
            collector_thread = threading.Thread(
                target=collector_thread_main,
                name="micro_ws_embedded",
                daemon=False,
            )
            ws_noise_buffer = io.StringIO()
            with contextlib.redirect_stdout(ws_noise_buffer), contextlib.redirect_stderr(ws_noise_buffer):
                collector_thread.start()
                try:
                    time.sleep(embedded_duration_sec)
                finally:
                    collector.stop()
                collector_thread.join(timeout=120)
            if collector_errors:
                raise RuntimeError(
                    f"Colector WS embebido terminó con error: {collector_errors[0]}"
                ) from collector_errors[0]
            if collector_thread.is_alive():
                print("Aviso: hilo del colector aún activo tras stop; revisa el registro (stdout o logs del entorno).")
            print(f"Colector WS embebido detenido tras {embedded_duration_sec}s.")

        # En modo embebido restringe a los símbolos de prueba; en modo observador lee el universo completo
        scope_embedded = LIVE_WS_EMBEDDED_RUN_SEC > 0
        where_clause = "WHERE symbol IN :syms" if scope_embedded else ""
        snapshot_params = {"syms": live_symbols} if scope_embedded else {}

        state_query = text(
            f"""
            SELECT symbol, last_agg_trade_id, last_trade_ts, active_bucket_start,
                   active_bucket_trade_count, last_finalized_bucket,
                   last_rest_backfill_ts, updated_at
            FROM {MICROSTRUCTURE_SCHEMA}.ws_live_state
            {where_clause}
            ORDER BY symbol
            """
        )
        staging_query = text(
            f"""
            SELECT symbol, COUNT(*) AS rows_staging,
                   MIN(transact_time) AS min_transact_time,
                   MAX(transact_time) AS max_transact_time
            FROM {MICROSTRUCTURE_SCHEMA}.ws_agg_trade_staging
            {where_clause}
            GROUP BY symbol
            ORDER BY symbol
            """
        )
        if scope_embedded:
            state_query = state_query.bindparams(bindparam("syms", expanding=True))
            staging_query = staging_query.bindparams(bindparam("syms", expanding=True))

        with live_engine.connect() as connection:
            micro_live_snapshot_state = pd.read_sql(
                state_query, connection, params=snapshot_params
            ).rename(columns=LIVE_SNAPSHOT_COL_STATE_ES)
            micro_live_snapshot_staging = pd.read_sql(
                staging_query, connection, params=snapshot_params
            ).rename(columns=LIVE_SNAPSHOT_COL_STAGING_ES)

        print(
            f"Snapshot live [{LIVE_SNAPSHOT_LABEL}] UTC "
            f"{datetime.now(timezone.utc).isoformat()}"
        )
        if LIVE_WS_EMBEDDED_RUN_SEC > 0:
            print(
                "(Modo embebido) Evidencia: ws_live_state + agregado ws_agg_trade_staging para los símbolos de prueba."
            )
        live_state_rows = len(micro_live_snapshot_state)
        live_staging_rows = (
            int(micro_live_snapshot_staging["filas"].sum())
            if not micro_live_snapshot_staging.empty
            else 0
        )
        if LIVE_WS_EMBEDDED_RUN_SEC > 0:
            print(
                f"Resumen (smoke): símbolos en el agregado de staging={len(micro_live_snapshot_staging)} | "
                f"total de filas de trades={live_staging_rows}"
            )
        else:
            print(
                f"Resumen: ws_live_state={live_state_rows} filas | "
                f"staging (suma filas)={live_staging_rows}"
            )
        print("ws_live_state - estado persistido por símbolo")
        display(micro_live_snapshot_state)
        print("\nws_agg_trade_staging - ocupación temporal agregada (snapshot)")
        display(micro_live_snapshot_staging)

        if LIVE_WS_EMBEDDED_RUN_SEC > 0:
            symbols_with_staging = (
                set(micro_live_snapshot_staging["símbolo"].astype(str))
                if not micro_live_snapshot_staging.empty
                else set()
            )
            missing_staging_symbols = [symbol for symbol in live_symbols if symbol not in symbols_with_staging]
            if missing_staging_symbols:
                raise RuntimeError(
                    "Prueba live no superada: sin trades en staging para símbolos "
                    + ", ".join(missing_staging_symbols)
                )
            zero_staging_symbols = micro_live_snapshot_staging[
                micro_live_snapshot_staging["filas"] < 1
            ]["símbolo"].astype(str).tolist()
            if zero_staging_symbols:
                raise RuntimeError(
                    "Prueba live no superada: staging con 0 filas para " + ", ".join(zero_staging_symbols)
                )
        elif live_state_rows == 0:
            raise RuntimeError(
                "Prueba live no superada: ws_live_state está vacío. "
                "Si usaste modo embebido, aumenta LIVE_WS_EMBEDDED_RUN_SEC o LIVE_WS_SYMBOLS; "
                "si colector externo, confirma misma BD y que micro-ws-live está en marcha."
            )

        if live_staging_rows == 0:
            print(
                "Aviso: staging sin filas en este instante (vaciado, descarga o símbolos sin actividad)."
            )
        if LIVE_WS_EMBEDDED_RUN_SEC > 0:
            if micro_live_snapshot_staging["max_timestamp"].notna().any():
                print("Aceptado: hay max_timestamp en staging (trades persistidos con marca temporal).")
            else:
                print("Aviso: max_timestamp todo NULL en el agregado mostrado.")
        elif micro_live_snapshot_state["ultimo_trade_ts"].notna().any():
            print("Aceptado: hay valor en `ultimo_trade_ts` (el colector recibe trades en esta BD).")
        else:
            print(
                "Aviso: toda la columna `ultimo_trade_ts` es nula (sin trades o colector recién iniciado)."
            )

        if (
            LIVE_WS_EMBEDDED_RUN_SEC > 0
            and LIVE_WS_EMBEDDED_CLEANUP_AFTER
            and live_symbols
        ):
            microstructure_db = settings.microstructure_data.db
            staging_table = microstructure_db.tables.ws_agg_trade_staging
            live_state_table = microstructure_db.tables.ws_live_state
            delete_staging_query = text(
                f"DELETE FROM {MICROSTRUCTURE_SCHEMA}.{staging_table} WHERE symbol IN :syms"
            ).bindparams(bindparam("syms", expanding=True))
            delete_live_state_query = text(
                f"DELETE FROM {MICROSTRUCTURE_SCHEMA}.{live_state_table} WHERE symbol IN :syms"
            ).bindparams(bindparam("syms", expanding=True))
            with live_engine.begin() as connection:
                deleted_staging = connection.execute(delete_staging_query, {"syms": live_symbols})
                deleted_live_state = connection.execute(delete_live_state_query, {"syms": live_symbols})
            print(
                f"Limpieza tras prueba embebida: filas borradas en staging={deleted_staging.rowcount}, "
                f"ws_live_state={deleted_live_state.rowcount} | símbolos={live_symbols}"
            )
    finally:
        live_engine.dispose()


Colector WS embebido: 1800s | inicio UTC 2026-06-04T00:56:56.533455+00:00 | símbolos=['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT'] | (sin finalización 4h -> fact; limpieza staging/estado al terminar)
Websocket connected
Colector WS embebido detenido tras 1800s.
Snapshot live [antes_o_despues_prueba_8h] UTC 2026-06-04T01:28:30.106611+00:00
(Modo embebido) Evidencia: ws_live_state + agregado ws_agg_trade_staging para los símbolos de prueba.
Resumen (smoke): símbolos en el agregado de staging=5 | total de filas de trades=491657
ws_live_state - estado persistido por símbolo


,símbolo,ultimo_id_agg_trade,ultimo_trade_ts,inicio_bucket_activo,trades_bucket_activo,ultimo_bucket_finalizado,ultimo_backfill_rest,actualizado_en
0,BNBUSDT,833230295,2026-06-04 01:28:29.756000+00:00,2026-06-04 00:00:00+00:00,0,None,2026-06-04 00:56:58.317136+00:00,2026-06-04 01:28:29.911134+00:00
1,BTCUSDT,3974064150,2026-06-04 01:28:29.634000+00:00,2026-06-04 00:00:00+00:00,0,None,2026-06-04 00:56:57.611493+00:00,2026-06-04 01:28:29.891229+00:00
2,ETHUSDT,2002244138,2026-06-04 01:28:29.842000+00:00,2026-06-04 00:00:00+00:00,0,None,2026-06-04 00:56:57.999994+00:00,2026-06-04 01:28:29.896427+00:00
3,SOLUSDT,657066025,2026-06-04 01:28:29.410000+00:00,2026-06-04 00:00:00+00:00,0,None,2026-06-04 00:56:58.944610+00:00,2026-06-04 01:28:29.906216+00:00
4,XRPUSDT,705435717,2026-06-04 01:28:27.868000+00:00,2026-06-04 00:00:00+00:00,0,None,2026-06-04 00:56:58.621521+00:00,2026-06-04 01:28:29.901076+00:00



ws_agg_trade_staging - ocupación temporal agregada (snapshot)


,símbolo,filas,min_timestamp,max_timestamp
0,BNBUSDT,54474,2026-06-04 00:29:50.968000+00:00,2026-06-04 01:28:29.756000+00:00
1,BTCUSDT,226308,2026-06-04 00:29:50.925000+00:00,2026-06-04 01:28:29.634000+00:00
2,ETHUSDT,164535,2026-06-04 00:29:50.938000+00:00,2026-06-04 01:28:29.842000+00:00
3,SOLUSDT,27769,2026-06-04 00:29:50.989000+00:00,2026-06-04 01:28:29.410000+00:00
4,XRPUSDT,18571,2026-06-04 00:29:51.018000+00:00,2026-06-04 01:28:27.868000+00:00


Aceptado: hay max_timestamp en staging (trades persistidos con marca temporal).
Limpieza tras prueba embebida: filas borradas en staging=491657, ws_live_state=5 | símbolos=['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT']


## 6. Exportación de artefactos para reproducibilidad

Por último se vuelcan a `reports/validation/microstructure/` un JSON consolidado y siete **CSV** (mismo `<stamp>`, p. ej. `20260603T233430Z` en esta corrida): `microstructure_validation_summary_`, `microstructure_validation_cobertura_`, `microstructure_validation_auditoria_`, `microstructure_validation_modos_`, `microstructure_validation_duplicados_`, `microstructure_validation_integridad_` y `microstructure_validation_parquet_inventory_` (prefijo común + sufijo `.csv`).

Indicadores resumidos en el JSON (`completitud_etl`, `duplicados_pk`, `integridad_imputaciones`, `inventario_parquet`):

- `histórico_detectado`: hay al menos un run batch histórico en `ingestion_runs`.
- `cero_duplicados_todos_símbolos`: contrato cero estricto sobre la PK `(symbol, bucket_4h)`.
- `inventario_parquet.archivos_encontrados / archivos_esperados`: confirma que el respaldo en disco existe para todos los símbolos.

Estos artefactos resumen de forma trazable la evidencia del ETL de microestructura. Los gráficos de la sección 5 son solo exploratorios (`plt.show()`); no se exportan a `reports/`.

**Nota.** Resultado global, tablas SQL, smoke *WebSocket* y rutas con `stamp` dependen del estado del proyecto al ejecutar el cuaderno; en otra corrida pueden cambiar.


In [6]:
# Escribe el paquete JSON y los CSV bajo reports/validation/microstructure
MICROSTRUCTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_paths = {
    "json": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_{RUN_STAMP}.json",
    "summary": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_summary_{RUN_STAMP}.csv",
    "cobertura": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_cobertura_{RUN_STAMP}.csv",
    "auditoria": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_auditoria_{RUN_STAMP}.csv",
    "modos": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_modos_{RUN_STAMP}.csv",
    "duplicados": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_duplicados_{RUN_STAMP}.csv",
    "integridad": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_integridad_{RUN_STAMP}.csv",
    "parquet_inventory": MICROSTRUCTURE_OUTPUT_DIR / f"microstructure_validation_parquet_inventory_{RUN_STAMP}.csv",
}

historico_ok = bool((not resumen_modos_df.empty) and (resumen_modos_df["modo"] == "histórico").any())
live_ok = bool((not resumen_modos_df.empty) and (resumen_modos_df["modo"] == "live").any())
duplicados_ok = bool(not duplicados_pk_df.empty and (duplicados_pk_df["duplicados_pk"] == 0).all())
integridad_ok = bool(not integridad_df.empty and len(integridad_df) == len(symbols_evidencia))
payload = {
    "metadata": run_meta,
    "fecha_objetivo_freeze_resuelta": MICROSTRUCTURE_FREEZE_COVERAGE_DATE,
    "target_bucket_utc": val._target_bucket_start(MICROSTRUCTURE_FREEZE_COVERAGE_DATE).isoformat(),
    "symbols_evidencia": list(symbols_evidencia),
    "resultado_global": "PASS" if accepted else "FAIL",
    "fallos_accionables": actionable_failures,
    "checks": summary_df.to_dict(orient="records"),
    "completitud_etl": {
        "histórico_detectado": historico_ok,
        "live_detectado": live_ok
    },
    "duplicados_pk": {
        "verificado": duplicados_ok,
        "cero_duplicados_todos_símbolos": duplicados_ok,
        "detalle": duplicados_pk_df.to_dict(orient="records") if not duplicados_pk_df.empty else [],
    },
    "integridad_imputaciones": {
        "verificado": integridad_ok,
        "detalle": integridad_df.to_dict(orient="records") if not integridad_df.empty else [],
    },
    "inventario_parquet": {
        "archivos_encontrados": int((parquet_inventory_df["tamaño_mb"].notna()).sum()) if not parquet_inventory_df.empty else 0,
        "archivos_esperados": len(parquet_inventory_df) if not parquet_inventory_df.empty else 0,
        "detalle": parquet_inventory_df.to_dict(orient="records") if not parquet_inventory_df.empty else [],
    },
    "nota_auditoria_failed_count_por_run": (
        "total_fallos_eventos en el CSV de modos suma failed_count de ingestion_runs "
        "(eventos fallidos en esa ejecución). Las columnas `ejecuciones_con_fallo` y `pct_ejecuciones_con_fallo` describen "
        "cuántas ejecuciones tuvieron al menos un fallo. Ninguna de estas métricas invalida por sí sola "
        "la fact agregada 4h si las comprobaciones E2E de cobertura y de calidad siguen en PASS."
    ),
}

export_paths["json"].write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
summary_df.to_csv(export_paths["summary"], index=False)
if not vista_cobertura_filas_df.empty:
    vista_cobertura_filas_df.to_csv(export_paths["cobertura"], index=False)
if not evidencia_auditoria_df.empty:
    evidencia_auditoria_df.to_csv(export_paths["auditoria"], index=False)
if not resumen_modos_df.empty:
    resumen_modos_df.to_csv(export_paths["modos"], index=False)
if not duplicados_pk_df.empty:
    duplicados_pk_df.to_csv(export_paths["duplicados"], index=False)
if not integridad_df.empty:
    integridad_df.to_csv(export_paths["integridad"], index=False)
if not parquet_inventory_df.empty:
    parquet_inventory_df.to_csv(export_paths["parquet_inventory"], index=False)

print(f"Resultado global: {'PASS' if accepted else 'FAIL'}")
print(f"Completitud histórico: {'OK' if historico_ok else 'PENDIENTE'}")
print(
    f"Live en ingestion_runs (opcional; evidencia operativa en §5): "
    f"{'OK' if live_ok else 'N/A (normal: live por WebSocket)'}")
print(f"Duplicados = 0 verificado: {'OK' if duplicados_ok else 'PENDIENTE'}")
print(f"Integridad por símbolo: {'OK' if integridad_ok else 'PENDIENTE'}")
print(f"Evidencia JSON: {export_paths['json']}")
print(f"Resumen de comprobaciones (CSV): {export_paths['summary']}")
if not vista_cobertura_filas_df.empty:
    print(f"Cobertura CSV: {export_paths['cobertura']}")
if not evidencia_auditoria_df.empty:
    print(f"Auditoría CSV: {export_paths['auditoria']}")
if not resumen_modos_df.empty:
    print(f"Modos histórico/live CSV: {export_paths['modos']}")
if not duplicados_pk_df.empty:
    print(f"Duplicados PK CSV: {export_paths['duplicados']}")
if not integridad_df.empty:
    print(f"Integridad/imputaciones CSV: {export_paths['integridad']}")
if not parquet_inventory_df.empty:
    print(f"Inventario Parquet CSV: {export_paths['parquet_inventory']}")

if engine:
    engine.dispose()


Resultado global: PASS
Completitud histórico: OK
Live en ingestion_runs (opcional; evidencia operativa en §5): N/A (normal: live por WebSocket)
Duplicados = 0 verificado: OK
Integridad por símbolo: OK
Evidencia JSON: /app/reports/validation/microstructure/microstructure_validation_20260604T005644Z.json
Resumen de comprobaciones (CSV): /app/reports/validation/microstructure/microstructure_validation_summary_20260604T005644Z.csv
Cobertura CSV: /app/reports/validation/microstructure/microstructure_validation_cobertura_20260604T005644Z.csv
Auditoría CSV: /app/reports/validation/microstructure/microstructure_validation_auditoria_20260604T005644Z.csv
Modos histórico/live CSV: /app/reports/validation/microstructure/microstructure_validation_modos_20260604T005644Z.csv
Duplicados PK CSV: /app/reports/validation/microstructure/microstructure_validation_duplicados_20260604T005644Z.csv
Integridad/imputaciones CSV: /app/reports/validation/microstructure/microstructure_validation_integridad_20260604